# Day 46: Simulate Continuous Batching

**Layer:** Implementation


## Concept Overview

Continuous batching inserts new requests at each decode step when a slot frees up. This eliminates the head-of-line blocking in static batching where a long request delays all new arrivals.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Simulating Continuous Batching

Continuous batching inserts new requests into the active batch at each decode step when slots free up, unlike static batching which waits for the full batch to complete.


In [ ]:
import heapq, numpy as np

def simulate_continuous_batching(arrivals, durations, max_batch=8, seed=42):
    np.random.seed(seed)
    queue = list(zip(arrivals, durations, range(len(arrivals))))
    queue.sort()  # sort by arrival time
    active = []  # (finish_time, req_id)
    t = 0
    completions = []
    step = 0
    while queue or active:
        # Remove completed requests
        active = [(ft, rid) for ft, rid in active if ft > t]
        # Fill batch with new arrivals
        while queue and queue[0][0] <= t and len(active) < max_batch:
            arr, dur, rid = queue.pop(0)
            active.append((t + dur, rid))
            completions.append({'id': rid, 'start': t, 'dur': dur})
        # Advance time by one decode step
        if active:
            t += 1
            for i, (ft, rid) in enumerate(active):
                active[i] = (ft, rid)
        elif queue:
            t = queue[0][0]  # jump to next arrival
        else:
            break
        step += 1
        if step > 10000: break
    return completions

np.random.seed(42)
n = 100
arrivals = np.cumsum(np.random.exponential(2, n))
durations = np.random.randint(5, 50, n)
results = simulate_continuous_batching(arrivals, durations)
latencies = [r['dur'] for r in results]
print(f'Simulated {n} requests with continuous batching:')
print(f'  Throughput: {n/max(arrivals)*..5:.1f} req/s estimate')
print(f'  Mean latency: {np.mean(latencies):.1f} steps')
print(f'  P99 latency:  {np.percentile(latencies,99):.1f} steps')


## 2. Static vs Continuous Batching Comparison

Static batching waits for all requests in a batch to complete. Continuous batching achieves higher GPU utilization by always keeping the batch full.


In [ ]:
import matplotlib.pyplot as plt

# Simulate utilization
np.random.seed(42)
T = 100
# Static batching: wait for batch to complete
static_util = []
batch_size = 8
for t in range(T):
    # GPU utilization: 1 if batch active, 0 if waiting for new batch
    in_batch = (t % 20) < 15  # 75% efficiency
    static_util.append(1.0 if in_batch else 0.0)
# Continuous batching: always at least one request
continuous_util = [min(1.0, 0.5 + np.random.exponential(0.3)) for _ in range(T)]
continuous_util = [min(1.0, u) for u in continuous_util]

plt.figure(figsize=(12,4))
plt.plot(static_util, label=f'Static batching (util={np.mean(static_util):.0%})', alpha=0.8)
plt.plot(continuous_util, label=f'Continuous batching (util={np.mean(continuous_util):.0%})', alpha=0.8)
plt.xlabel('Decode step'); plt.ylabel('GPU Utilization')
plt.title('Static vs Continuous Batching GPU Utilization')
plt.legend(); plt.grid(True)
plt.savefig('continuous_batching.png', dpi=100, bbox_inches='tight')
plt.show()


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Continuous batching inserts new requests at each decode step when a slot frees up.
- The head-of-line blocking problem in static batching: a single long request (1000 tokens) holds the entire batch hostage, starving 7 other users. Continuous batching routes around this by scheduling at iteration granularity..
- Day 46 implementation complete.
